In [7]:
##Bayes Theorem
# Given probabilities
P_repay = 0.6
P_not_repay = 0.4

P_high_income_given_repay = 0.7
P_high_income_given_not_repay = 0.2

# Total probability of High Income
P_high_income = (P_high_income_given_repay * P_repay) + \
                (P_high_income_given_not_repay * P_not_repay)

# Bayes Theorem
P_repay_given_high_income = (P_high_income_given_repay * P_repay) / P_high_income

print("Probability customer will repay given high income:",
      P_repay_given_high_income)

Probability customer will repay given high income: 0.84


In [ ]:
##Naïve Bayes classifier

In [18]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

# Check class distribution
unique, counts = np.unique(y, return_counts=True)

print("Original Dataset Distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u} count:", c)

Original Dataset Distribution:
Class 0 count: 50
Class 1 count: 50
Class 2 count: 50


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # IMPORTANT
)

# Check training distribution
unique, counts = np.unique(y_train, return_counts=True)

print("Training Distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u} count:", c)

print("\nTotal Training Samples:", len(y_train))
print("Total Testing Samples:", len(y_test))

Training Distribution:
Class 0 count: 40
Class 1 count: 40
Class 2 count: 40

Total Training Samples: 120
Total Testing Samples: 30


In [21]:
#Separate Training Data by Class
def separate_by_class(X, y):
    separated = {}
    for i in range(len(y)):
        label = y[i]
        if label not in separated:
            separated[label] = []
        separated[label].append(X[i])
    return separated

separated = separate_by_class(X_train, y_train)

# Check class counts again
for label in separated:
    print(f"Class {label} samples:", len(separated[label]))

Class 0 samples: 40
Class 2 samples: 40
Class 1 samples: 40


In [22]:
#Calculate Prior Probabilities
total_samples = len(y_train)

priors = {}

for label in separated:
    priors[label] = len(separated[label]) / total_samples

print("Prior Probabilities:")
for label in priors:
    print(f"P(Class {label}) =", priors[label])

Prior Probabilities:
P(Class 0) = 0.3333333333333333
P(Class 2) = 0.3333333333333333
P(Class 1) = 0.3333333333333333


In [26]:
#Calculate Mean and Variance for Each Feature (Per Class)
def summarize_dataset(dataset):
    summaries = []
    
    # zip(*dataset) converts rows to columns
    for feature in zip(*dataset):
        mean = np.mean(feature)
        var = np.var(feature)
        summaries.append((mean, var))
    
    return summaries

summaries = {}

for class_value, rows in separated.items():
    summaries[class_value] = summarize_dataset(rows)

In [27]:
print("Mean and Variance for Class 0:\n")

for i, (mean, var) in enumerate(summaries[0]):
    print(f"Feature {i}: Mean = {mean:.3f}, Variance = {var:.3f}")

Mean and Variance for Class 0:

Feature 0: Mean = 4.985, Variance = 0.093
Feature 1: Mean = 3.415, Variance = 0.155
Feature 2: Mean = 1.478, Variance = 0.025
Feature 3: Mean = 0.255, Variance = 0.013


In [28]:
print("\nMean and Variance for Class 1:\n")
for i, (mean, var) in enumerate(summaries[1]):
    print(f"Feature {i}: Mean = {mean:.3f}, Variance = {var:.3f}")


Mean and Variance for Class 1:

Feature 0: Mean = 5.930, Variance = 0.222
Feature 1: Mean = 2.750, Variance = 0.093
Feature 2: Mean = 4.253, Variance = 0.191
Feature 3: Mean = 1.320, Variance = 0.034


In [31]:
#Apply Gaussian Formula to ONE Test Sample (Real Probability Calculation)

#
sample = X_test[0]

print("Test Sample:")
print(sample)

print("\nActual Class:", y_test[0])

Test Sample:
[4.4 3.  1.3 0.2]

Actual Class: 0


In [33]:
#
def gaussian_probability(x, mean, var):
    exponent = np.exp(-((x - mean) ** 2) / (2 * var))
    return (1 / np.sqrt(2 * np.pi * var)) * exponent

In [34]:
probabilities = {}

for class_value, class_summaries in summaries.items():
    
    # Start with prior
    probabilities[class_value] = priors[class_value]
    
    print(f"\nClass {class_value}")
    print("Prior:", priors[class_value])
    
    # Multiply Gaussian probabilities
    for i in range(len(class_summaries)):
        mean, var = class_summaries[i]
        x = sample[i]
        
        prob = gaussian_probability(x, mean, var)
        
        print(f"Feature {i} → Probability: {prob}")
        
        probabilities[class_value] *= prob
    
    print("Final Probability for Class", class_value, "=", probabilities[class_value])


Class 0
Prior: 0.3333333333333333
Feature 0 → Probability: 0.20710464842890516
Feature 1 → Probability: 0.5814441561486375
Feature 2 → Probability: 1.3452894602915482
Feature 3 → Probability: 3.116955001511914
Final Probability for Class 0 = 0.16831502044651056

Class 2
Prior: 0.3333333333333333
Feature 0 → Probability: 0.0028318608688624783
Feature 1 → Probability: 1.1398320352943352
Feature 2 → Probability: 3.5776712953588675e-13
Feature 3 → Probability: 5.4239479382367264e-11
Final Probability for Class 2 = 2.087889283800667e-26

Class 1
Prior: 0.3333333333333333
Feature 0 → Probability: 0.004307868905744023
Feature 1 → Probability: 0.9348378888202462
Feature 2 → Probability: 1.1878671746575665e-10
Feature 3 → Probability: 2.221114909403362e-08
Final Probability for Class 1 = 3.541738060443276e-21


In [35]:
prediction = max(probabilities, key=probabilities.get)

print("\nPredicted Class:", prediction)
print("Actual Class:", y_test[0])


Predicted Class: 0
Actual Class: 0


In [32]:
#Predict Entire Test Set & Compute Accuracy

In [36]:
#Create Prediction Function
def predict(row):
    probabilities = {}

    for class_value, class_summaries in summaries.items():
        probabilities[class_value] = priors[class_value]

        for i in range(len(class_summaries)):
            mean, var = class_summaries[i]
            x = row[i]
            probabilities[class_value] *= gaussian_probability(x, mean, var)

    return max(probabilities, key=probabilities.get)

In [37]:
#Predict All Test Samples
predictions = []

for row in X_test:
    result = predict(row)
    predictions.append(result)

print("First 10 Predictions:", predictions[:10])
print("First 10 Actual:", y_test[:10])

First 10 Predictions: [np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(2), np.int64(1)]
First 10 Actual: [0 2 1 1 0 1 0 0 2 1]


In [38]:
#Calculate Accuracy
correct = 0

for i in range(len(y_test)):
    if predictions[i] == y_test[i]:
        correct += 1

accuracy = correct / len(y_test)

print("Accuracy:", accuracy)

Accuracy: 0.9666666666666667


In [ ]:
#KNN

In [39]:
#Define Euclidean Distance
from math import sqrt

def euclidean_distance(row1, row2):
    distance = 0
    for i in range(len(row1)):
        distance += (row1[i] - row2[i]) ** 2
    return sqrt(distance)

In [40]:
#Test Distance on One Example
sample = X_test[0]

print("Test Sample:", sample)

Test Sample: [4.4 3.  1.3 0.2]


In [41]:
for i in range(5):
    dist = euclidean_distance(sample, X_train[i])
    print(f"Distance to Train Sample {i}:", dist)

Distance to Train Sample 0: 0.14142135623730948
Distance to Train Sample 1: 3.60416425818802
Distance to Train Sample 2: 4.414748010928823
Distance to Train Sample 3: 0.5567764362830022
Distance to Train Sample 4: 3.1559467676119


In [42]:
#Get Nearest Neighbors
def get_neighbors(train_X, train_y, test_row, k):
    distances = []

    for i in range(len(train_X)):
        dist = euclidean_distance(test_row, train_X[i])
        distances.append((dist, train_y[i]))

    # Sort by distance
    distances.sort(key=lambda x: x[0])

    # Select k nearest
    neighbors = distances[:k]
    
    return neighbors

In [43]:
#Predict One Sample
k = 3

neighbors = get_neighbors(X_train, y_train, sample, k)

print("Nearest Neighbors (distance, class):")
for n in neighbors:
    print(n)

Nearest Neighbors (distance, class):
(0.14142135623730948, np.int64(0))
(0.244948974278318, np.int64(0))
(0.29999999999999954, np.int64(0))


In [44]:
def predict_knn(train_X, train_y, test_row, k):
    neighbors = get_neighbors(train_X, train_y, test_row, k)
    
    output_values = [neighbor[1] for neighbor in neighbors]
    
    prediction = max(set(output_values), key=output_values.count)
    
    return prediction

In [45]:

prediction = predict_knn(X_train, y_train, sample, k)

print("Predicted Class:", prediction)
print("Actual Class:", y_test[0])

Predicted Class: 0
Actual Class: 0


In [46]:
predictions_knn = []

for row in X_test:
    result = predict_knn(X_train, y_train, row, k)
    predictions_knn.append(result)

print("First 10 Predictions:", predictions_knn[:10])
print("First 10 Actual:", y_test[:10])

First 10 Predictions: [np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(2), np.int64(1)]
First 10 Actual: [0 2 1 1 0 1 0 0 2 1]


In [47]:
correct = 0

for i in range(len(y_test)):
    if predictions_knn[i] == y_test[i]:
        correct += 1

accuracy_knn = correct / len(y_test)

print("k-NN Accuracy:", accuracy_knn)

k-NN Accuracy: 1.0
